In [1]:
import pandas as pd
df = pd.read_csv("data/stock prices.csv", usecols=["date", "close", "high", "low", "open", "adjClose", "adjHigh", "adjLow", "adjOpen"]).dropna()
df.head()

,date,close,high,low,open,adjClose,adjHigh,adjLow,adjOpen
0,2015-05-27 00:00:00+00:00,132.045,132.260,130.05,130.34,121.682558,121.880685,119.844118,120.111360
1,2015-05-28 00:00:00+00:00,131.780,131.950,131.10,131.86,121.438354,121.595013,120.811718,121.512076
2,2015-05-29 00:00:00+00:00,130.280,131.450,129.90,131.23,120.056069,121.134251,119.705890,120.931516
3,2015-06-01 00:00:00+00:00,130.535,131.390,130.05,131.20,120.291057,121.078960,119.844118,120.903870
4,2015-06-02 00:00:00+00:00,129.960,130.655,129.32,129.86,119.761181,120.401640,119.171406,119.669029


In [2]:
df = df.sort_values(by="date").set_index(keys="date", drop=True)
df.head(15)

,close,high,low,open,adjClose,adjHigh,adjLow,adjOpen
date,,,,,,,,
2015-05-27 00:00:00+00:00,132.045,132.260,130.050,130.340,121.682558,121.880685,119.844118,120.111360
2015-05-28 00:00:00+00:00,131.780,131.950,131.100,131.860,121.438354,121.595013,120.811718,121.512076
2015-05-29 00:00:00+00:00,130.280,131.450,129.900,131.230,120.056069,121.134251,119.705890,120.931516
2015-06-01 00:00:00+00:00,130.535,131.390,130.050,131.200,120.291057,121.078960,119.844118,120.903870
2015-06-02 00:00:00+00:00,129.960,130.655,129.320,129.860,119.761181,120.401640,119.171406,119.669029
2015-06-03 00:00:00+00:00,130.120,130.940,129.900,130.660,119.908625,120.664274,119.705890,120.406248
2015-06-04 00:00:00+00:00,129.360,130.580,128.910,129.580,119.208267,120.332526,118.793582,119.411002
2015-06-05 00:00:00+00:00,128.650,129.690,128.360,129.500,118.553986,119.512370,118.286744,119.337280
2015-06-08 00:00:00+00:00,127.800,129.210,126.830,128.900,117.770691,119.070039,116.876813,118.784366


In [3]:
# Splitting the dataset
test_split = 0.2
train_df = df.iloc[ : int(df.shape[0] * (1 - test_split)), : ]
test_df = df.iloc[int(df.shape[0] * (1 - test_split)): , : ]

X_train, y_train = train_df.drop(columns=['close']), train_df['close']
X_test, y_test = test_df.drop(columns=['close']), test_df['close']

In [4]:
# Shapes
print(X_train.shape)
print(X_test.shape)

(1006, 7)
(252, 7)


In [5]:
# Scaling the input features
from sklearn.preprocessing import MinMaxScaler

scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_x.fit(X_train)

scaler_y = MinMaxScaler(feature_range=(0, 1))
scaler_y.fit(y_train.to_frame())

# Transforming train and test input features → NumPy Array
X_train = scaler_x.transform(X_train)
X_test = scaler_x.transform(X_test)

y_train = scaler_y.transform(y_train.to_frame())
y_test = scaler_y.transform(y_test.to_frame())

In [6]:
# Shapes
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(1006, 7)
(252, 7)
(1006, 1)
(252, 1)


In [7]:
# Dataset Preperation for Deep Learning model
import numpy as np

def prepare_dataset(dataset: np.array, timestep: int = 1) -> pd.DataFrame:
    dataset = pd.DataFrame(dataset)
    periods = list(range(1, timestep + 1))
    return dataset.shift(periods).iloc[timestep + 1:].to_numpy()

# timestep —→ Hyperparameter (More would be great)
timestep = 50
X_train = prepare_dataset(dataset=X_train, timestep=timestep)
X_test = prepare_dataset(dataset=X_test, timestep=timestep)

y_train = y_train[timestep + 1 : ]
y_test = y_test[timestep + 1 : ]

In [8]:
# Shapes
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(955, 350)
(201, 350)
(955, 1)
(201, 1)


In [9]:
# Model Building
import torch
from torch.nn import Module, RNN, Linear

class CustomModel(Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.rnn = RNN(input_size=input_size, hidden_size=25, num_layers=1, nonlinearity="tanh", batch_first=True, bidirectional=False)
        self.linear = Linear(in_features=25, out_features=1)
        
    def forward(self, X_train):
        _, final = self.rnn(X_train)
        y_pred = self.linear(final)
        return y_pred

In [10]:
# Training parameters
learning_rate = 0.001
epochs = 2500

In [11]:
# Loss function and optimizer
model = CustomModel(input_size=X_train.shape[1] // timestep)

criterion = torch.nn.modules.loss.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(params = model.parameters())

In [12]:
# Training pipeline
for i in range(epochs):

    # Conterting data from NumPy array to Tensor
    X_train_cp = torch.tensor(X_train, dtype=torch.float32).reshape(X_train.shape[0], timestep, X_train.shape[1] // timestep) # NOTE —→ Hard coded
    y_train_cp = torch.tensor(y_train, dtype=torch.float32)

    # Initializing all gradients to zero
    optimizer.zero_grad()

    # Forward propogation
    y_pred = model(X_train_cp)

    # Loss calculation
    loss = criterion(y_pred.reshape(-1), y_train_cp.reshape(-1))

    # Gradient calculation
    loss.backward()

    # Updating weights
    optimizer.step()

    if (i + 1) % 100 == 0:
        print(f"Epoch {i + 1} → Loss = {loss.item()}")

Epoch 100 → Loss = 0.00683749420568347
Epoch 200 → Loss = 0.005712322890758514
Epoch 300 → Loss = 0.004945719614624977
Epoch 400 → Loss = 0.004103929735720158
Epoch 500 → Loss = 0.0022255845833569765
Epoch 600 → Loss = 0.0026343590579926968
Epoch 700 → Loss = 0.0029849186539649963
Epoch 800 → Loss = 0.0042267171666026115
Epoch 900 → Loss = 0.0013218835229054093
Epoch 1000 → Loss = 0.0012195551535114646
Epoch 1100 → Loss = 0.002698602620512247
Epoch 1200 → Loss = 0.0012722600949928164
Epoch 1300 → Loss = 0.0010637437226250768
Epoch 1400 → Loss = 0.0009996446315199137
Epoch 1500 → Loss = 0.0014500472461804748
Epoch 1600 → Loss = 0.0011900310637429357
Epoch 1700 → Loss = 0.0011414834298193455
Epoch 1800 → Loss = 0.001064057694748044
Epoch 1900 → Loss = 0.0008653388358652592
Epoch 2000 → Loss = 0.0008269519894383848
Epoch 2100 → Loss = 0.0013843713095411658
Epoch 2200 → Loss = 0.0010703930165618658
Epoch 2300 → Loss = 0.003998303320258856
Epoch 2400 → Loss = 0.0009080137824639678
Epoch 250

In [13]:
model.eval()

CustomModel(
  (rnn): RNN(7, 25, batch_first=True)
  (linear): Linear(in_features=25, out_features=1, bias=True)
)

In [14]:
with torch.no_grad():
    # Conterting data from NumPy array to Tensor
    X_test_cp = torch.tensor(X_test, dtype=torch.float32).reshape(X_test.shape[0], timestep, X_test.shape[1] // timestep) # NOTE —→ Hard coded
    y_test_cp = torch.tensor(y_test, dtype=torch.float32)

    # Forward propogation
    y_pred = model(X_test_cp)  

In [15]:
import plotly.graph_objects as go
import numpy as np

# Ensure data is 1D arrays for Plotly
# If y_test or y_pred are 2D (e.g., shape (n_samples, 1)), flatten them
actual_prices = scaler_y.inverse_transform(y_test).flatten()
predicted_prices = scaler_y.inverse_transform(y_pred.reshape(1, -1)).flatten()

# Create the x-axis (Days). 
# Assuming len(actual_prices) represents the number of days.
days = np.arange(len(actual_prices))

# Create the figure
fig = go.Figure()

# Add Actual Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=actual_prices,
    mode='lines',
    name='Actual Stock Prices',
    line=dict(color='blue', width=2)
))

# Add Predicted Prices trace
fig.add_trace(go.Scatter(
    x=days,
    y=predicted_prices,
    mode='lines',
    name='Predicted Stock Prices',
    line=dict(color='orange', width=2, dash='dash')
))

# Update layout
fig.update_layout(
    title='Stock Price Prediction',
    xaxis_title='Days',
    yaxis_title='Price',
    hovermode='x unified',
    template='plotly_white',
    width=1000,  # Adjust width to match your matplotlib figsize preference
    height=500
)

# Show the chart
fig.show()